# 08 - Balanced Comparison

## Objective

In notebook 07 only Logistic Regression had class weights, so the comparison was not fair. LightGBM also gave a strange result on the easy set. Here we:

1. Give class weights to all models and compare them again on the easy set (E1) and the hard sets (H1, H2).
2. Check why LightGBM failed.
3. Check if a model finds the frauds that the rule `ratio_is_one` misses.

After this notebook we choose the final model. Only the validation set is used. The test set is still untouched.

## Imports and Data Loading

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score

sys.path.append('..')
from src.features import config
from src.data.splits import temporal_masks, TRAIN_END_STEP, VAL_END_STEP
from src.models.feature_sets import FEATURE_SETS, HARD_SETS, add_ratio_features, prepare_linear
from src.models.evaluation import evaluate, bootstrap_pr_auc

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
SEED = config.RANDOM_SEED

In [2]:
table = add_ratio_features(pd.read_parquet(config.PROCESSED_PATH))
in_scope = (table['type_CASH_OUT'] + table['type_TRANSFER']) == 1
data = table[in_scope].reset_index(drop=True)
del table

tr_mask, va_mask, _ = temporal_masks(data, TRAIN_END_STEP, VAL_END_STEP)
tr, va = data[tr_mask], data[va_mask]
ytr, yva = tr['isFraud'].to_numpy(), va['isFraud'].to_numpy()
SETS = {'E1_core': FEATURE_SETS['S1_core'], 'H1_no_ratio': HARD_SETS['H1_no_ratio'],
        'H2_no_origin_balance': HARD_SETS['H2_no_origin_balance']}
print(f'train {len(tr):,} rows ({ytr.sum():,} frauds), validation {len(va):,} rows ({yva.sum():,} frauds)')

train 2,645,779 rows (5,561 frauds), validation 86,155 rows (1,286 frauds)


## 1. All models with class weights

Same models as notebook 07, but every model is balanced. PR-AUC comes with a 95% interval.

In [3]:
results, scores_store = [], {}

def make_models():
    models = {
        'Logistic Regression': (make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=300, class_weight='balanced')), True),
        'Decision Tree (depth 6)': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=100,
                                    class_weight='balanced', random_state=SEED), False),
        'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20,
                          max_samples=0.2, class_weight='balanced_subsample', n_jobs=-1,
                          random_state=SEED), False),
        'Gradient Boosting': (HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                              class_weight='balanced', random_state=SEED), False),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = (LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                              class_weight='balanced', random_state=SEED, verbose=-1), False)
    except ImportError:
        print('lightgbm is not installed, skipped')
    return models

def run(model, name, set_name, linear=False):
    cols = SETS[set_name]
    Xtr, Xva = tr[cols], va[cols]
    if linear:
        Xtr, Xva = prepare_linear(Xtr), prepare_linear(Xva)
    start = time.time()
    model.fit(Xtr, ytr)
    secs = time.time() - start
    scores = model.predict_proba(Xva)[:, 1]
    row = {'model': name, 'features': set_name, 'seconds': round(secs, 1)}
    row.update(evaluate(yva, scores))
    row['pr_auc_low'], row['pr_auc_high'] = bootstrap_pr_auc(yva, scores)
    results.append(row)
    scores_store[(name, set_name)] = scores

for set_name in SETS:
    for name, (model, linear) in make_models().items():
        run(model, name, set_name, linear=linear)

comparison = pd.DataFrame(results)
show_cols = ['model', 'features', 'pr_auc', 'pr_auc_low', 'pr_auc_high', 'roc_auc', 'best_f1',
             'recall_at_p80', 'recall_at_p95', 'alerts_per_10k_at_p95', 'seconds']
comparison.sort_values(['features', 'pr_auc'], ascending=[True, False])[show_cols].round(4)

,model,features,pr_auc,pr_auc_low,pr_auc_high,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95,seconds
4,LightGBM,E1_core,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,157.0425,31.6
2,Random Forest,E1_core,1.0000,1.0000,1.0000,1.0000,0.9996,1.0000,1.0000,157.0425,36.2
1,Decision Tree (depth 6),E1_core,1.0000,1.0000,1.0000,1.0000,0.9988,1.0000,1.0000,149.8462,23.6
3,Gradient Boosting,E1_core,0.9999,0.9996,1.0000,1.0000,0.9992,1.0000,0.9992,156.9265,22.0
0,Logistic Regression,E1_core,0.9998,0.9995,1.0000,1.0000,0.9992,1.0000,0.9984,156.8104,4.2
8,Gradient Boosting,H1_no_ratio,0.9305,0.9223,0.9403,0.9984,0.8556,0.8577,0.7613,119.5520,57.3
9,LightGBM,H1_no_ratio,0.9275,0.9197,0.9370,0.9984,0.8545,0.8367,0.7722,121.2930,55.0
7,Random Forest,H1_no_ratio,0.8587,0.8425,0.8739,0.9925,0.8282,0.7893,0.7302,114.6770,83.6
6,Decision Tree (depth 6),H1_no_ratio,0.7019,0.6800,0.7241,0.9821,0.7169,0.4448,0.3888,60.2403,24.4
5,Logistic Regression,H1_no_ratio,0.1519,0.1421,0.1643,0.9422,0.2630,0.0000,0.0000,0.0000,8.1


**Observation:**
- Easy set (E1): all five models are between 0.9998 and 1.0000. We cannot tell them apart. LightGBM is now perfect.
- H1 (no ratio features): Gradient Boosting (0.931) and LightGBM (0.928) are the same inside the interval. Random Forest is lower (0.859), the tree gets 0.702 and Logistic Regression 0.152. Class weights lifted the boosting models a lot (0.67 to 0.93 for Gradient Boosting), but much less for the tree and the forest.
- H2 (no origin balance): everything drops. LightGBM (0.490), Gradient Boosting (0.483) and the tree (0.456) overlap. Random Forest is clearly worse (0.357) and Logistic Regression is at 0.100. At 95% precision the best recall is only about 0.20.
- So boosting is first or tied on every set, and the linear model fails when the ratio features are removed. Without the origin balance the problem is really hard.

## 2. Why did LightGBM fail on the easy set?

In notebook 07 it had a PR-AUC of 0.46 on E1 while the other models had 1.0. We look at the data types and the score on the training set, and try two other settings.

In [4]:
try:
    from lightgbm import LGBMClassifier
    cols = SETS['E1_core']
    print(tr[cols].dtypes.to_string())
    print()
    configs = {
        'default (notebook 07)': dict(n_estimators=300, learning_rate=0.05, num_leaves=31),
        'class weights': dict(n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight='balanced'),
        'class weights, smaller leaves': dict(n_estimators=300, learning_rate=0.1, num_leaves=15,
                                              min_child_samples=5, class_weight='balanced'),
    }
    rows = []
    for label, params in configs.items():
        m = LGBMClassifier(random_state=SEED, verbose=-1, **params).fit(tr[cols], ytr)
        rows.append({'setting': label,
                     'pr_auc_train': round(average_precision_score(ytr, m.predict_proba(tr[cols])[:, 1]), 4),
                     'pr_auc_validation': round(average_precision_score(yva, m.predict_proba(va[cols])[:, 1]), 4)})
    lgbm_check = pd.DataFrame(rows)
    lgbm_check
except ImportError:
    lgbm_check = None
    print('lightgbm is not installed')

amount_log                 float32
oldbalanceOrg_log          float32
oldbalanceDest_log         float32
amount_to_balance_ratio    float32
zero_old_balance_orig         int8
ratio_gap_to_one           float32
ratio_is_one                  int8
type_TRANSFER                 int8



**Observation:** With the notebook 07 settings, LightGBM had a PR-AUC of only 0.21 on the **training** set too. So it was not a data problem. It simply did not learn the rare class without weights. With class weights, train and validation are both 1.0. The strange result in notebook 07 is explained.

## 3. Does a model catch what the rule misses?

The rule `ratio_is_one` misses 22 frauds on validation. We take the rows that the rule does **not** flag, rank them by model score, and count how many of the 22 missed frauds are in the top rows. This is done on the hard sets (the model has no ratio feature), with the best model of each set.

In [5]:
flag = (va['ratio_is_one'] == 1).to_numpy()
missed_mask = (~flag) & (yva == 1)
print('Frauds missed by the rule:', int(missed_mask.sum()), '| rows not flagged by the rule:', int((~flag).sum()))

rows = []
for set_name in ['E1_core', 'H1_no_ratio', 'H2_no_origin_balance']:
    sub = comparison[comparison['features'] == set_name].sort_values('pr_auc', ascending=False)
    best_name = sub.iloc[0]['model']
    s = scores_store[(best_name, set_name)]
    idx = np.flatnonzero(~flag)
    order = idx[np.argsort(-s[idx])]
    for k in [10, 50, 100, 500]:
        top = order[:k]
        rows.append({'set': set_name, 'model': best_name, 'extra_alerts': k,
                     'missed_frauds_found': int(yva[top].sum()),
                     'extra_false_alerts': int(k - yva[top].sum())})
pd.DataFrame(rows)

Frauds missed by the rule: 22 | rows not flagged by the rule: 84891


,set,model,extra_alerts,missed_frauds_found,extra_false_alerts
0,E1_core,LightGBM,10,10,0
1,E1_core,LightGBM,50,22,28
2,E1_core,LightGBM,100,22,78
3,E1_core,LightGBM,500,22,478
4,H1_no_ratio,Gradient Boosting,10,10,0
5,H1_no_ratio,Gradient Boosting,50,19,31
6,H1_no_ratio,Gradient Boosting,100,19,81
7,H1_no_ratio,Gradient Boosting,500,19,481
8,H2_no_origin_balance,LightGBM,10,8,2
9,H2_no_origin_balance,LightGBM,50,19,31


**Observation:** The model finds what the rule misses.
- E1: LightGBM finds all 22 missed frauds in its 50 top-ranked rows (28 false alerts). The first 10 are all frauds.
- H1: Gradient Boosting finds 19 of 22 in the top 50.
- H2 (no origin balance): LightGBM also finds 19 of 22 in the top 50 and 20 in the top 500. So the destination features carry real signal.
- 50 extra alerts on 86,155 rows is about 6 per 10,000 transactions. A few frauds (2 or 3) stay hard to find on the harder sets.

## 4. Conclusion and decisions

**Findings**
- On the easy set all models are equal (PR-AUC about 1.0). The ratio effect of PaySim makes the problem trivial.
- On the harder sets, boosting (Gradient Boosting, LightGBM) is the best or tied. Logistic Regression and Random Forest fall behind.
- With class weights, LightGBM works. The earlier failure was a missing-weights problem.
- A model adds value on top of the rule: it finds the frauds with a new destination that the rule misses.

**Decisions**
- Final features: `E1_core` (8 features). All of them come from one transaction (amount, origin balance, destination balance, type), so the API does not need any transaction history. The destination history gave no clear gain on the easy set and would force us to keep state in the API.
- Final model: LightGBM with class weights. It ties with Gradient Boosting on performance, and it gives native per-transaction contributions, which we use for the explanations.
- No more model comparison and no long hyperparameter search. The models are tied, so tuning would not change the decision.

**Next notebook:** `09_final_model.ipynb`. Threshold on validation, one evaluation on the test set, global and local explanations, and saving the model.